# 1. End-to-End Evaluation: Generated Candidates → Reranker → Blend

The other reranking notebooks score **logged** slates. This notebook instead reranks the
**candidate generator's own output**, which is how the two stages would run together:

1. Take each validation impression's generated candidates (popularity block ranks 1–500,
   content block ranks 501–1250)
2. Replay the validation timeline so every candidate gets its causal ranking features as
   of that impression, then score it with a trained `HybridSequenceReranker`
3. Compare the generator order, the reranker order, and blends of the two

**Labels are incomplete.** A generated article that was never shown has unknown relevance:
it counts as "not a known click", not as a negative. Report candidate recall alongside the
ranking metrics. Choose candidate budgets and blend weights on validation only.

## 1.1 Load Packages

In [ ]:
import json
import sys
import time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
import pyarrow as pa
import pyarrow.parquet as pq
import torch
from IPython.display import display
from torch import nn

## 1.2 Set Up Paths

Set `ENVIRONMENT` to `"colab"` to read the project and data from Google Drive. The
repository supplies the shared evaluator; the data folder holds the prepared features
and receives every checkpoint, score file and metric table this notebook writes.

In [ ]:
ENVIRONMENT = "local"  # "local" or "colab"

if ENVIRONMENT == "colab":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/drive/MyDrive/BT4222-Group-6-")
    DATA_ROOT = Path("/content/drive/MyDrive/v3-mindsmall")
else:
    PROJECT_ROOT = Path("/Users/numkoos/Downloads/BT4222-Group-6-")
    DATA_ROOT = Path("/Users/numkoos/Downloads/v3-mindsmall")

# Ranking features, click labels and causal user histories from feature engineering
RUN = DATA_ROOT / "4_Feature_Engineering" / "artifacts" / "prequential-all"

# Frozen article embeddings from the representation stage
REPRESENTATIONS = DATA_ROOT / "3_Representations" / "artifacts"

# Checkpoints, score files and metric tables written by the reranking notebooks
ARTIFACTS = DATA_ROOT / "5_Models" / "5_02_Reranking" / "artifacts"
RESULTS = DATA_ROOT / "results"

# The shared evaluator is the only project module the reranking notebooks import
sys.path.insert(0, str(PROJECT_ROOT / "5_Models" / "5_02_Reranking"))
from rr_evaluation import compare_results, rank_logged_slate, score_and_evaluate

In [ ]:
# Causal feature replay comes from the project's feature-engineering package
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from newsrec.common import DEFAULT_SETTINGS
from newsrec.pipeline import replay_batches, time_horizon
from newsrec.ranking import TemporalState, build_features, feature_schema
from newsrec.representations import Bundle

from rr_evaluation import evaluate_retrieved_candidates

# 2. Experiment Configuration

`POPULARITY_TOP = 200` and `CONTENT_TOP = 200` keep generator ranks 1–200 and 501–700.
Set `RUN_SCORING = True` only when that score file does not exist yet; the timeline
replay takes several minutes.

In [ ]:
CANDIDATES = Path("/Users/numkoos/Downloads/recommendations_validation_top1250.parquet")
MODEL_DIR = ARTIFACTS / "hybrid_lstm_lambda_ndcg5_full_v1"   # any RR_hybrid_sequence_mlp output
POPULARITY_TOP = 200        # generator ranks 1..POPULARITY_TOP (max 500)
CONTENT_TOP = 200           # generator ranks 501..500 + CONTENT_TOP (max 750)
RUN_SCORING = False
BLEND_WEIGHTS = np.round(np.arange(0.0, 1.01, 0.05), 2)   # reranker share of the blend
HISTORY_CAP = 50
BATCH_SIZE = 4096
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

suffix = f"top{POPULARITY_TOP}" if CONTENT_TOP == 0 else f"pop{POPULARITY_TOP}_content{CONTENT_TOP}"
SCORE_PATH = MODEL_DIR / f"retrieved_validation_{suffix}.parquet"
print(f"Score file: {SCORE_PATH}")

# 3. Load the Trained Reranker

The classes below match `RR_hybrid_sequence_mlp.ipynb` exactly, so its saved weights load
directly.

In [ ]:
text_vectors = np.load(REPRESENTATIONS / "text64.npy", mmap_mode="r")
EMBEDDING_DIM = text_vectors.shape[1]

In [ ]:
class ArticleEncoder(nn.Module):
    """Map article rows to their frozen 64-dimensional text embeddings (row 0 is padding)."""

    def __init__(self, vectors):
        super().__init__()
        self.embedding = nn.Embedding.from_pretrained(
            torch.from_numpy(np.array(vectors, dtype=np.float32, copy=True)),
            freeze=True,
            padding_idx=0,
        )

    def forward(self, article_rows):
        return self.embedding(article_rows)                  # [..., 64]


class HistorySequenceEncoder(nn.Module):
    """Run an LSTM or GRU over the chronological click history and keep the last valid step."""

    def __init__(self, article_encoder, rnn_type):
        super().__init__()
        self.article_encoder = article_encoder
        rnn_class = nn.LSTM if rnn_type == "lstm" else nn.GRU
        self.recurrent_layer = rnn_class(EMBEDDING_DIM, EMBEDDING_DIM, batch_first=True)

    def forward(self, history_rows):
        # history_rows: [batch, history], right-padded with 0
        outputs, _ = self.recurrent_layer(self.article_encoder(history_rows))
        lengths = history_rows.ne(0).sum(dim=1)
        last_step = lengths.clamp(min=1) - 1
        user = outputs[torch.arange(len(history_rows), device=history_rows.device), last_step]
        return user * lengths.ne(0).unsqueeze(1)             # zero vector for an empty history

In [ ]:
class CandidateScorer(nn.Module):
    """MLP over [tabular features ; candidate text ; user vector] that returns one logit."""

    def __init__(self, tabular_width, hidden_dims=(256, 64), dropout=0.20):
        super().__init__()
        layers, width = [], tabular_width + 2 * EMBEDDING_DIM
        for hidden in hidden_dims:
            layers.extend([nn.Linear(width, hidden), nn.ReLU(), nn.Dropout(dropout)])
            width = hidden
        layers.append(nn.Linear(width, 1))
        self.layers = nn.Sequential(*layers)

    def forward(self, tabular, candidate_text, user):
        return self.layers(torch.cat([tabular, candidate_text, user], dim=1)).squeeze(-1)


class HybridSequenceReranker(nn.Module):
    """Article encoder → recurrent user encoder → tabular candidate scorer."""

    def __init__(self, vectors, tabular_width, rnn_type):
        super().__init__()
        self.article_encoder = ArticleEncoder(vectors)
        self.user_encoder = HistorySequenceEncoder(self.article_encoder, rnn_type)
        self.candidate_scorer = CandidateScorer(tabular_width)

    def encode_history(self, history_rows):
        """Encode each impression's history once; candidates of that impression reuse it."""
        return self.user_encoder(history_rows)

    def forward(self, tabular, candidate_rows, user):
        return self.candidate_scorer(tabular, self.article_encoder(candidate_rows), user)

In [ ]:
def encode_histories(impression_ids, profiles):
    """Return right-padded chronological histories holding the newest HISTORY_CAP clicks."""
    histories = np.zeros((len(impression_ids), HISTORY_CAP), dtype=np.int64)
    for index, uid in enumerate(impression_ids):
        history = profiles[uid][-HISTORY_CAP:]
        if len(history):
            histories[index, :len(history)] = history
    return histories

In [ ]:
metadata = json.loads((MODEL_DIR / "model_metadata.json").read_text())
checkpoint = torch.load(MODEL_DIR / "model.pt", map_location="cpu", weights_only=True)
preprocessor = joblib.load(MODEL_DIR / "preprocessor.joblib")
FEATURES = checkpoint["features"]

# The scorer's first layer sees [tabular ; candidate ; user], so subtract the two text vectors
tabular_width = checkpoint["state_dict"]["candidate_scorer.layers.0.weight"].shape[1] - 2 * EMBEDDING_DIM
model = HybridSequenceReranker(text_vectors, tabular_width, metadata["rnn_type"])
model.load_state_dict(checkpoint["state_dict"])
model.to(DEVICE).eval()

saved_schema = json.loads((RUN / "feature_schema.json").read_text())
if feature_schema(DEFAULT_SETTINGS)["model_columns"] != saved_schema["model_columns"] or FEATURES != saved_schema["model_columns"]:
    raise ValueError("The feature code, feature run and trained model disagree on the feature schema")
print(f"Loaded {metadata['rnn_type'].upper()} reranker trained with {metadata.get('objective', 'bce_bpr')}")

# 4. Score Generated Candidates

The validation timeline is replayed in timestamp order. Each impression's candidates get
features from the state **before** that impression; candidates never update the state.
Rows are scored in chunks of about 20,000 and written to a partial file that is renamed
only when every impression has been covered.

In [ ]:
def score_retrieved_candidates():
    """Replay validation, build causal features for generated candidates and score them."""
    candidate_filter = (pl.col("cutoff") <= POPULARITY_TOP) | (
        (pl.col("cutoff") >= 501) & (pl.col("cutoff") <= 500 + CONTENT_TOP)
    )
    candidates = (
        pl.scan_parquet(CANDIDATES).filter(candidate_filter)
        .select("impression_uid", "news_code", "cutoff").collect().to_pandas()
        .sort_values("impression_uid").reset_index(drop=True)
    )
    if candidates.duplicated(["impression_uid", "news_code"]).any():
        raise ValueError("Duplicate candidates in a slate")
    rows_by_impression = candidates.groupby("impression_uid", sort=False).indices

    profiles = pl.read_parquet(RUN / "request_profiles_validation" / "*.parquet",
                               columns=["impression_uid", "full_history_rows"])
    histories = dict(zip(profiles["impression_uid"].to_list(), profiles["full_history_rows"].to_list()))

    bundle = Bundle(DATA_ROOT, DEFAULT_SETTINGS)
    state = TemporalState(bundle, *time_horizon(DATA_ROOT), DEFAULT_SETTINGS)
    partial_path = SCORE_PATH.with_suffix(".partial.parquet")
    writer, pending, impressions, started = None, [], 0, time.monotonic()

    def flush():
        """Score the pending feature rows and append them to the partial score file."""
        nonlocal writer, pending
        if not pending:
            return
        frame = pd.concat(pending, ignore_index=True)
        pending = []

        uids = frame["impression_uid"].drop_duplicates().tolist()
        with torch.no_grad():
            users = model.encode_history(torch.from_numpy(encode_histories(uids, histories)).to(DEVICE)).cpu().numpy()
        user_rows = frame["impression_uid"].map(pd.Series(np.arange(len(uids)), index=uids)).to_numpy()
        tabular = preprocessor.transform(frame[FEATURES]).astype(np.float32, copy=False)
        article_rows = frame["article_row"].to_numpy(np.int64)

        scores = []
        with torch.no_grad():
            for start in range(0, len(frame), BATCH_SIZE):
                stop = start + BATCH_SIZE
                logits = model(
                    torch.from_numpy(tabular[start:stop]).to(DEVICE),
                    torch.from_numpy(article_rows[start:stop]).to(DEVICE),
                    torch.from_numpy(users[user_rows[start:stop]]).to(DEVICE),
                )
                scores.append(torch.sigmoid(logits).cpu().numpy())
        frame["score"] = np.concatenate(scores)

        table = pa.Table.from_pandas(frame[["impression_uid", "news_code", "cutoff", "score"]], preserve_index=False)
        if writer is None:
            writer = pq.ParquetWriter(partial_path, table.schema, compression="zstd")
        writer.write_table(table)

    try:
        for timestamp, requests, events in replay_batches(DATA_ROOT, DEFAULT_SETTINGS, splits=("train", "validation")):
            state.begin(requests, timestamp)
            validation_requests = requests[requests["split"].eq("validation")]
            if len(validation_requests):
                selected = [rows_by_impression[uid] for uid in validation_requests["impression_uid"] if uid in rows_by_impression]
                if len(selected) != len(validation_requests):
                    raise ValueError("A validation impression has no generated candidate list")
                pairs = candidates.iloc[np.concatenate(selected)]
                features = build_features(validation_requests, pairs, state, bundle, DEFAULT_SETTINGS, "retrieved")
                pending.append(features.merge(pairs, on=["impression_uid", "news_code"], validate="one_to_one"))
                impressions += len(validation_requests)
                if sum(len(frame) for frame in pending) >= 20_000:
                    flush()
            state.finish(requests, events, timestamp, {"train", "validation"})
        flush()
    finally:
        if writer is not None:
            writer.close()
        state.close()

    if impressions != len(rows_by_impression):
        raise ValueError(f"Scored {impressions:,} impressions but {len(rows_by_impression):,} have candidates")
    partial_path.rename(SCORE_PATH)
    print(f"Scored {impressions:,} impressions in {(time.monotonic() - started) / 60:.1f} minutes")


if RUN_SCORING:
    score_retrieved_candidates()
if not SCORE_PATH.exists():
    raise FileNotFoundError("Set RUN_SCORING = True or choose an existing score file.")

# 5. Compare Generator, Reranker and Blends

Each candidate has two signals: its generator rank (`cutoff`, turned into the prior
$-\log(1 + \text{cutoff})$) and its reranker probability (`score`). Both are standardised
**within each impression** before blending, so neither wins only because of its scale:

$$s_{blend} = w\, z_{reranker} + (1 - w)\, z_{generator}$$

In [ ]:
candidates = pl.scan_parquet(SCORE_PATH)
counts = candidates.group_by("impression_uid").len().collect()
expected = POPULARITY_TOP + CONTENT_TOP
if counts["len"].min() != expected or counts["len"].max() != expected:
    raise ValueError(f"Expected {expected} candidates per impression")
print(f"{counts.height:,} impressions with {expected} candidates each")


def within_impression_z(expression):
    """Standardise a score within each impression (constant slates become 0)."""
    centred = expression - expression.mean().over("impression_uid")
    return (centred / expression.std().over("impression_uid")).fill_nan(0).fill_null(0)


generator_prior = -pl.col("cutoff").cast(pl.Float64).log1p()
z_generator = within_impression_z(generator_prior)
z_reranker = within_impression_z(pl.col("score"))


def evaluate_blends(frame, prefix=""):
    """Known-click metrics for the generator, the reranker and every blend weight."""
    methods = {f"{prefix}generator": generator_prior, f"{prefix}reranker": pl.col("score")}
    for weight in BLEND_WEIGHTS:
        methods[f"{prefix}blend_{weight:.2f}"] = weight * z_reranker + (1 - weight) * z_generator
    results = [
        evaluate_retrieved_candidates(
            frame.with_columns(expression.alias("final_score")),
            feature_run=RUN,
            score_column="final_score",
            model_name=name,
        )
        for name, expression in methods.items()
    ]
    return pl.concat(results).sort(["ndcg_at_5", "mrr"], descending=True)


results = evaluate_blends(candidates)
display(results.select("model", "candidate_recall", "mrr", "ndcg_at_5", "ndcg_at_10", "recall_at_10"))
print(f"Validation-selected configuration: {results['model'][0]}")

# 6. Observed-Overlap Diagnostic

Keeping only generated candidates that were actually shown isolates **ordering quality
among labelled candidates**. Its denominator still contains every known click, so this is
a conditional diagnostic, not an end-to-end recall metric.

In [ ]:
labels = pl.scan_parquet(RUN / "labels_validation" / "*.parquet").select("impression_uid", "news_code")
shown = candidates.join(labels, on=["impression_uid", "news_code"], how="inner")
shown_counts = shown.group_by("impression_uid").len().collect()
print(f"Shown candidates per impression: mean={shown_counts['len'].mean():.1f}, median={shown_counts['len'].median():.0f}")

overlap_results = evaluate_blends(shown, prefix="overlap_")
display(overlap_results.select("model", "candidate_recall", "mrr", "ndcg_at_5", "ndcg_at_10"))